# 02. Exterior-Only Virtual Scan

The source point cloud contains labelled samples from many fragment surfaces, including surfaces that would be hidden inside the pile. This notebook removes interior/occluded points by keeping only points visible from exterior viewpoints. The result approximates a real LiDAR survey where the scanner only observes the outside of the muckpile.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from scanning.occlusion_model import exterior_points_from_viewpoints
from scanning.scan_noise import reduce_density, add_range_noise
from io_utils.export_pointcloud import save_labelled_npz, save_ascii_ply

SCAN_DIR = ROOT / 'data' / 'scanned_pointclouds'
TABLE_DIR = ROOT / 'outputs' / 'tables'
FIGURE_DIR = ROOT / 'outputs' / 'figures'
for path in [SCAN_DIR, TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

PARAMS = {
    'angular_resolution_deg': 0.22,
    'density_keep_fraction': 0.78,
    'range_noise_sigma_m': 0.0015,
    'random_seed': 2026,
}

full_cloud = np.load(SCAN_DIR / 'synthetic_rockpile_full_labelled_pointcloud.npz')
points = full_cloud['points_xyz']
labels = full_cloud['fragment_ids']

bounds_min = points.min(axis=0)
bounds_max = points.max(axis=0)
center = 0.5 * (bounds_min + bounds_max)
span = bounds_max - bounds_min
radius = float(np.linalg.norm(span[:2]))
z_mid = float(center[2] + 0.45 * span[2])
viewpoints = [
    center + np.array([ 1.9 * radius, 0.0, z_mid - center[2]]),
    center + np.array([-1.9 * radius, 0.0, z_mid - center[2]]),
    center + np.array([0.0,  1.9 * radius, z_mid - center[2]]),
    center + np.array([0.0, -1.9 * radius, z_mid - center[2]]),
    center + np.array([0.0, 0.0, 2.1 * span[2]]),
]
print(points.shape, len(np.unique(labels)), viewpoints)

In [ ]:
exterior_points, exterior_labels, exterior_indices = exterior_points_from_viewpoints(
    points,
    labels,
    viewpoint_xyz_list=viewpoints,
    angular_resolution_deg=PARAMS['angular_resolution_deg'],
)
scan_points, scan_labels = reduce_density(
    exterior_points,
    exterior_labels,
    keep_fraction=PARAMS['density_keep_fraction'],
    random_seed=PARAMS['random_seed'],
)
scan_points = add_range_noise(
    scan_points,
    viewpoint_xyz=viewpoints[0],
    sigma_m=PARAMS['range_noise_sigma_m'],
    random_seed=PARAMS['random_seed'] + 1,
)

save_labelled_npz(
    SCAN_DIR / 'synthetic_rockpile_exterior_only_scan.npz',
    scan_points,
    scan_labels,
    extra={
        'exterior_source_indices': exterior_indices,
        'viewpoints_xyz': np.asarray(viewpoints),
        'angular_resolution_deg': np.array(PARAMS['angular_resolution_deg']),
    },
)
save_ascii_ply(SCAN_DIR / 'synthetic_rockpile_exterior_only_scan.ply', scan_points, scan_labels)

scan_summary = pd.DataFrame([{
    'n_full_points': int(len(points)),
    'n_exterior_points_before_density_reduction': int(len(exterior_points)),
    'n_final_scan_points': int(len(scan_points)),
    'n_fragments_full_cloud': int(len(np.unique(labels))),
    'n_fragments_visible_after_exterior_filter': int(len(np.unique(exterior_labels))),
    'n_fragments_final_scan': int(len(np.unique(scan_labels))),
    **PARAMS,
}])
scan_summary.to_csv(TABLE_DIR / 'exterior_scan_summary.csv', index=False)
scan_summary

In [ ]:
rng = np.random.default_rng(PARAMS['random_seed'])
idx_full = rng.choice(len(points), size=min(65000, len(points)), replace=False)
idx_scan = rng.choice(len(scan_points), size=min(65000, len(scan_points)), replace=False)

fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(points[idx_full, 0], points[idx_full, 1], points[idx_full, 2], c=labels[idx_full], s=0.7, cmap='tab20')
ax1.set_title('Original full labelled point cloud')
ax1.set_xlabel('x [m]'); ax1.set_ylabel('y [m]'); ax1.set_zlabel('z [m]')
ax1.view_init(elev=22, azim=-58)

ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(scan_points[idx_scan, 0], scan_points[idx_scan, 1], scan_points[idx_scan, 2], c=scan_labels[idx_scan], s=1.2, cmap='tab20')
vp = np.asarray(viewpoints)
ax2.scatter(vp[:, 0], vp[:, 1], vp[:, 2], c='red', marker='^', s=45)
ax2.set_title('Exterior-only scan after occlusion filtering')
ax2.set_xlabel('x [m]'); ax2.set_ylabel('y [m]'); ax2.set_zlabel('z [m]')
ax2.view_init(elev=22, azim=-58)
fig.tight_layout()
figure_path = FIGURE_DIR / 'synthetic_rockpile_exterior_scan_preview.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print(figure_path)